In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
df_raw = pd.read_csv("../data/processed_dataset.csv")
df=pd.read_csv("../data/processed_dataset.csv")

In [ ]:
print("\n→ df.info()")
df_raw.info()
print("\n→ Missing Values")
print(df_raw.isnull().sum())
print("\n→ df.describe()")
print(df_raw.describe())

In [ ]:
df = df.drop(columns=['Unnamed: 0','name'],errors='ignore')

In [ ]:
df = df.drop('blood_pressure', axis=1)

In [ ]:
cols = ['obesity_group','bp_category']

dummy = pd.get_dummies(df[cols]).astype(int)
df = pd.concat([df,dummy],axis=1)
df = df.drop(columns=cols)

In [ ]:
# Preprocessing dataset
print("\n→ df.info()")
df.info()
print("\n→ Missing Values")
print(df.isnull().sum())
print("\n→ df.describe()")
print(df.describe())

In [ ]:
print("Feature engineering completed on both datasets!")
print("Final cleaned datasets ready!")
print(f"Raw cleaned shape: {df_raw.shape}")
print(f"Processed cleaned shape: {df.shape}")
print("Sample from Processed cleaned dataset:")
print(df.head(5))

In [ ]:
#Visualizing
plt.figure(figsize=(15, 10))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of DataFrame')
plt.show()

In [ ]:
numerical_cols = ['age', 'sugar_percentage', 'glucose_percentage', 'cholesterol_percentage', 'obesity_percentage', 'heart_rate', 'health_risk_score']
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['age', 'sugar_percentage', 'glucose_percentage', 'cholesterol_percentage', 'obesity_percentage', 'heart_rate', 'health_risk_score']
plt.figure(figsize=(18, 12))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop('has_eye_disease', axis=1)
y = df['has_eye_disease']

print("Columns in X after dropping 'blood_pressure' and 'has_eye_disease':")
print(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
cv_auc = cross_val_score(
    LogisticRegression(solver='liblinear'),
    X,
    y,
    cv=5,
    scoring='roc_auc'
)

print("Mean CV ROC-AUC:", cv_auc.mean())

In [ ]:
model = LogisticRegression(random_state=42, solver='liblinear') # liblinear is good for small datasets and binary classification
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Logistic Regression')
plt.show()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

In [ ]:
model_performance = {}
print("Initialized a dictionary to store model performance metrics.")

In [ ]:
model_name = 'RandomForestClassifier'
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

model_performance[model_name] = {
    'Accuracy': accuracy_rf,
    'Precision': precision_rf,
    'Recall': recall_rf,
    'F1-Score': f1_rf,
    'ROC AUC': roc_auc_rf
}

print(f"--- {model_name} Metrics ---")
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")
print(f"ROC AUC: {roc_auc_rf:.4f}")

cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix for {model_name}')
plt.show()

In [ ]:
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

feat_imp.head(10)

In [ ]:
model_name = 'GradientBoostingClassifier'
gbc_model = GradientBoostingClassifier(random_state=42)
gbc_model.fit(X_train, y_train)

y_pred_gbc = gbc_model.predict(X_test)
y_proba_gbc = gbc_model.predict_proba(X_test)[:, 1]

accuracy_gbc = accuracy_score(y_test, y_pred_gbc)
precision_gbc = precision_score(y_test, y_pred_gbc)
recall_gbc = recall_score(y_test, y_pred_gbc)
f1_gbc = f1_score(y_test, y_pred_gbc)
roc_auc_gbc = roc_auc_score(y_test, y_proba_gbc)

model_performance[model_name] = {
    'Accuracy': accuracy_gbc,
    'Precision': precision_gbc,
    'Recall': recall_gbc,
    'F1-Score': f1_gbc,
    'ROC AUC': roc_auc_gbc
}

print(f"--- {model_name} Metrics ---")
print(f"Accuracy: {accuracy_gbc:.4f}")
print(f"Precision: {precision_gbc:.4f}")
print(f"Recall: {recall_gbc:.4f}")
print(f"F1-Score: {f1_gbc:.4f}")
print(f"ROC AUC: {roc_auc_gbc:.4f}")

cm_gbc = confusion_matrix(y_test, y_pred_gbc)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_gbc, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix for {model_name}')
plt.show()

In [ ]:
model_comparison_df = pd.DataFrame.from_dict(model_performance, orient='index')
print(model_comparison_df)